# 🌸 MakeupLLM 训练
Qwen2.5-VL-7B + LoRA 微调，妆容风格识别

运行前：右上角 **Settings** → **Accelerator** → **GPU T4 x2**

In [ ]:
# Cell 1: 安装依赖
!pip install -q llamafactory[metrics] qwen-vl-utils modelscope
!pip install -q flash-attn --no-build-isolation
print('✅ 依赖安装完成')

In [ ]:
# Cell 2: 下载模型
from modelscope import snapshot_download
model_path = snapshot_download('Qwen/Qwen2.5-VL-7B-Instruct', local_dir='models/Qwen2.5-VL-7B')
print(f'✅ 模型下载完成: {model_path}')

In [ ]:
# Cell 3: 上传训练数据
# 在 Kaggle 左侧面板点击 + Add Data，上传以下文件：
#   - train.json
#   - eval.json
#   - dataset_info.json
#   - makeup_images.zip
# 上传后数据路径为 /kaggle/input/makeup-data/
import os, shutil, json

# Kaggle 数据路径（上传后自动挂载）
KAGGLE_INPUT = '/kaggle/input'
DATA_DIR = 'data'
os.makedirs(f'{DATA_DIR}/makeup_images', exist_ok=True)

# 查找上传的数据
for d in os.listdir(KAGGLE_INPUT):
    src = os.path.join(KAGGLE_INPUT, d)
    if os.path.isdir(src):
        for f in os.listdir(src):
            if f.endswith('.json'):
                shutil.copy(os.path.join(src, f), f'{DATA_DIR}/{f}')
                print(f'  ✅ {f}')
            elif f.endswith('.zip'):
                !cd {DATA_DIR} && unzip -q '{os.path.join(src, f)}' && rm '{f}'
                print(f'  ✅ 解压 {f}')

with open(f'{DATA_DIR}/train.json') as f:
    train = json.load(f)
print(f'\n训练集: {len(train)} 条')
print(f'图片数: {len(os.listdir(f"{DATA_DIR}/makeup_images"))} 张')

In [ ]:
# Cell 4: 克隆 LLaMA-Factory 并配置
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
!cd LLaMA-Factory && pip install -e ".[metrics]" -q

!cp data/train.json LLaMA-Factory/data/
!cp data/eval.json LLaMA-Factory/data/
!cp data/dataset_info.json LLaMA-Factory/data/
!cp -r data/makeup_images LLaMA-Factory/data/
print('✅ LLaMA-Factory 配置完成')

In [ ]:
# Cell 5: 创建训练配置
import yaml

config = {
    'model_name_or_path': 'models/Qwen2.5-VL-7B',
    'template': 'qwen2_vl',
    'trust_remote_code': True,
    'stage': 'sft',
    'do_train': True,
    'finetuning_type': 'lora',
    'lora_target': 'all',
    'lora_rank': 16,
    'lora_alpha': 16,
    'dataset': 'makeup_vl_train',
    'eval_dataset': 'makeup_vl_eval',
    'cutoff_len': 2048,
    'overwrite_cache': True,
    'preprocessing_num_workers': 4,
    'output_dir': 'saves/makeup-llm/lora',
    'logging_steps': 10,
    'save_steps': 100,
    'save_total_limit': 3,
    'plot_loss': True,
    'overwrite_output_dir': True,
    'per_device_train_batch_size': 2,
    'gradient_accumulation_steps': 8,
    'learning_rate': 1.0e-4,
    'num_train_epochs': 3.0,
    'lr_scheduler_type': 'cosine',
    'warmup_ratio': 0.1,
    'bf16': True,
    'flash_attn': 'fa2',
    'gradient_checkpointing': True,
}

with open('LLaMA-Factory/train_makeup.yaml', 'w') as f:
    yaml.dump(config, f, allow_unicode=True, default_flow_style=False)
print('✅ 训练配置已创建')

In [ ]:
# Cell 6: 开始训练！
import datetime
print(f'🚀 训练开始: {datetime.datetime.now()}')
print('=' * 50)

!cd LLaMA-Factory && llamafactory-cli train train_makeup.yaml

print('=' * 50)
print(f'✅ 训练完成: {datetime.datetime.now()}')

In [ ]:
# Cell 7: 打包模型供下载
!tar czf MakeupLLM-7B-lora.tar.gz -C saves/makeup-llm/lora .
print('✅ 模型已打包: MakeupLLM-7B-lora.tar.gz')
print('在右侧 Output 面板下载')